In [4]:
# Importação das bibliotecas necessárias
import numpy as np
import pandas as pd
import random

In [5]:
# -----------------------------
# 1. MATRIZ DE AVALIAÇÕES
# -----------------------------
# Linhas: usuários
# Colunas: filmes
# Valores: nota de 1 a 5 (0 significa que o usuário não avaliou o filme)

ratings_matrix = pd.DataFrame([
    [4, 5, 0, 0, 0],  # U1
    [0, 4, 2, 0, 0],  # U2
    [3, 0, 4, 0, 0],  # U3
    [0, 5, 0, 1, 0],  # U4
    [0, 0, 2, 0, 3],  # U5
], columns=["M1", "M2", "M3", "M4", "M5"], index=["U1", "U2", "U3", "U4", "U5"])

print("📊 Matriz original de avaliações:\n")
print(ratings_matrix)

📊 Matriz original de avaliações:

    M1  M2  M3  M4  M5
U1   4   5   0   0   0
U2   0   4   2   0   0
U3   3   0   4   0   0
U4   0   5   0   1   0
U5   0   0   2   0   3


In [6]:


# ---------------------------------
# 2. FATORAÇÃO DE MATRIZ COM SVD
# ---------------------------------
# Substituímos os 0s temporariamente por médias de linha para aplicar o SVD
R = ratings_matrix.to_numpy().astype(float)
mask = R == 0
row_means = np.true_divide(R.sum(1), (R != 0).sum(1))
R_filled = R.copy()
for i in range(R.shape[0]):
    R_filled[i, mask[i]] = row_means[i]

# Aplicamos SVD: R ≈ U @ S @ V.T
U, S, VT = np.linalg.svd(R_filled, full_matrices=False)

# Reduzimos o rank (opcional): usamos apenas k componentes principais
k = 2
U_k = U[:, :k]
S_k = np.diag(S[:k])
VT_k = VT[:k, :]

# Reconstruímos a matriz de predições
R_pred = np.dot(np.dot(U_k, S_k), VT_k)

print("\n🔍 Matriz de notas previstas:\n")
print(pd.DataFrame(np.round(R_pred, 2), index=ratings_matrix.index, columns=ratings_matrix.columns))


🔍 Matriz de notas previstas:

      M1    M2    M3    M4    M5
U1  4.11  4.96  4.33  4.48  4.63
U2  2.81  3.68  2.87  2.61  3.07
U3  3.16  3.65  3.37  3.69  3.60
U4  3.04  5.09  2.77  1.11  2.97
U5  2.26  2.65  2.41  2.60  2.57


In [ ]:
# ---------------------------------------------
# 3. RECOMENDAÇÕES PERSONALIZADAS
# ---------------------------------------------
def get_top_recommendations(user_index, R_true, R_pred, movie_names, n=3):
    """
    Retorna os top-N filmes recomendados para um usuário específico,
    ignorando os que ele já avaliou.
    """
    already_rated = R_true[user_index] > 0
    scores = R_pred[user_index]
    recommendations = [
        (movie_names[i], scores[i])
        for i in range(len(scores)) if not already_rated[i]
    ]
    recommendations.sort(key=lambda x: x[1], reverse=True)
    return recommendations[:n]

# Lista de filmes menos populares que queremos inserir
rare_movies = [("M6", 3.5), ("M7", 4.0)]

# Simulando recomendações para todos os usuários
print("\n🎯 Recomendações com inserção estratégica:\n")
for i, user in enumerate(ratings_matrix.index):
    recs = get_top_recommendations(i, R, R_pred, ratings_matrix.columns.tolist(), n=3)

    # Inserção estratégica (aleatória) de um filme raro
    insert_pos = random.randint(0, len(recs))  # posição aleatória para inserir
    rare_movie = random.choice(rare_movies)
    recs.insert(insert_pos, (rare_movie[0] + " ★ promovido", rare_movie[1]))

    # Exibindo
    print(f"\n🔸 Recomendações para {user}:")
    for movie, score in recs:
        print(f" - {movie}: {round(score, 2)}")





🎯 Recomendações com inserção estratégica:


🔸 Recomendações para U1:
 - M5: 4.63
 - M4: 4.48
 - M3: 4.33
 - M7 ★ promovido: 4.0

🔸 Recomendações para U2:
 - M5: 3.07
 - M1: 2.81
 - M4: 2.61
 - M6 ★ promovido: 3.5

🔸 Recomendações para U3:
 - M4: 3.69
 - M2: 3.65
 - M6 ★ promovido: 3.5
 - M5: 3.6

🔸 Recomendações para U4:
 - M1: 3.04
 - M6 ★ promovido: 3.5
 - M5: 2.97
 - M3: 2.77

🔸 Recomendações para U5:
 - M2: 2.65
 - M4: 2.6
 - M1: 2.26
 - M7 ★ promovido: 4.0
